In [14]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema.output_parser import StrOutputParser
from langchain_groq import ChatGroq
from langchain.schema.runnable import RunnablePassthrough

In [2]:
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

In [3]:
pdf_url= "https://s1.q4cdn.com/806093406/files/doc_financials/2025/q4/Q4-FY25_Press-Release_FINAL.pdf"
loader= PyPDFLoader(pdf_url)
data= loader.load()

In [4]:
text_splitter= RecursiveCharacterTextSplitter(chunk_size= 200, chunk_overlap= 50)
text_chunks= text_splitter.split_documents(data)

In [5]:
embeddings=HuggingFaceEmbeddings(model_name= "BAAI/bge-small-en")

In [6]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [7]:
retriever=vectorstore.as_retriever()

In [8]:
query = "What is the revenue generated?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
Total selling and administrative expense  4,148  4,088  1 % 16,088  16,576  -3 %
% of revenues  37.4 %  32.4 %  34.7 %  32.3 %
Interest expense (income), net  (22)  (53)  —  (107)  (161)  —
--------------------------------------------------
Document 2:
operating segment.
3 Corporate revenues primarily consist of foreign currency hedge gains and losses related to revenues generated by entities within the
--------------------------------------------------
Document 3:
Gross margin  40.3 %  44.7 %  42.7 %  44.6 %
Demand creation expense  1,253  1,091  15 % 4,689  4,285  9 %
Operating overhead expense  2,895  2,997  -3 % 11,399  12,291  -7 %
--------------------------------------------------
Document 4:
Revenues $ 11,097 $ 12,606  -12 %$ 46,309 $ 51,362  -10 %
Cost of sales  6,628  6,972  -5 % 26,519  28,475  -7 %
Gross profit  4,469  5,634  -21 % 19,790  22,887  -14 %
--------------------------------------------------


In [9]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [10]:
prompt=ChatPromptTemplate.from_template(template)

In [12]:
output_parser=StrOutputParser()

In [13]:
llm= ChatGroq(model= "openai/gpt-oss-20b")

In [16]:
rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [17]:
rag_chain.invoke("tell me about revenue generated")

'The company reported a decline in revenue for the fourth quarter of fiscal 2025.  \nQ4 revenue fell to $11.10\u202fbillion from $12.61\u202fbillion year‑ago, a 12\u202f% decrease.  \nFor the full fiscal year, revenue was $46.31\u202fbillion versus $51.36\u202fbillion in the prior year, a 10\u202f% drop.  \nDespite the revenue decline, the company maintained a gross margin of 40.3\u202f% in Q4.  \nOperating expenses, including selling, general and administrative costs, were $4.15\u202fbillion in Q4, or 37.4\u202f% of revenue.  \nThe company’s interest expense was $22\u202fmillion in Q4, compared with $53\u202fmillion in the prior year.  \nThese figures illustrate a downward trend in sales across both the quarter and the full year.  \nNo other revenue‑specific details are provided in the retrieved excerpts.'